The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [12]:
import subprocess, sys
pkgs = ["datasets", "plotly", "scikit-learn", "torch", "torchvision",
        "numpy", "nbformat", "ipywidgets", "umap-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs + ["-q"])

0

In [13]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [14]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train", verification_mode="no_checks")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000

vocab = [word for word, _ in counts.most_common(V)]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
corpus = [word2idx[t] for t in tokens if t in word2idx]

print(f"vocab size: {len(vocab)}")
print(f"corpus length after filtering: {len(corpus)}")
print(f"sample vocab: {vocab[:10]}")

vocab size: 8000
corpus length after filtering: 276042
sample vocab: ['the', 'of', 'and', 'in', 'to', 'a', 'was', 's', 'that', 'as']


## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [15]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)
        self.output = nn.Linear(dim, vocab_size)
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        embedded = self.center(center_ids)
        return self.output(embedded)

In [16]:
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

epoch_losses = []
for epoch in range(epochs):
    perm = np.random.permutation(len(pairs))
    shuffled = pairs[perm]

    total_loss = 0.0
    n_batches = 0
    for start in range(0, len(shuffled), B):
        batch = shuffled[start : start + B]
        center_ids = torch.tensor(batch[:, 0], dtype=torch.long)
        context_ids = torch.tensor(batch[:, 1], dtype=torch.long)

        logits = model(center_ids)
        loss = loss_fn(logits, context_ids)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        n_batches += 1

    avg = total_loss / n_batches
    epoch_losses.append(avg)
    print(f"epoch {epoch+1}/{epochs}  loss={avg:.4f}")

emb = model.center.weight.detach().cpu().numpy()
print(f"\nemb shape: {emb.shape}")

epoch 1/3  loss=6.9859
epoch 2/3  loss=6.7126
epoch 3/3  loss=6.5748

emb shape: (8000, 64)


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [17]:
import subprocess, sys

try:
    import sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "-q"])

from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]
X = emb[:N]

pca = PCA(n_components=3, random_state=0)
pca3 = pca.fit_transform(X)
print(f"PCA done: {pca3.shape}")

try:
    import umap
    reducer = umap.UMAP(n_components=3, random_state=0, n_neighbors=15, min_dist=0.1)
    umap3 = reducer.fit_transform(X)
    print(f"UMAP done: {umap3.shape}")
except ImportError:
    umap3 = None
    print("umap-learn not installed, skipping UMAP.")

PCA done: (1500, 3)


/Users/rahulyalamanchili/Library/Python/3.10/lib/python/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done: (1500, 3)


In [18]:
import plotly.graph_objects as go

def plot_embeddings(coords, words, query=None, neighbor_set=None):
    neighbor_set = neighbor_set or set()
    colors, sizes = [], []
    for w in words:
        if w == query:
            colors.append("red")
            sizes.append(8)
        elif w in neighbor_set:
            colors.append("orange")
            sizes.append(6)
        else:
            colors.append("steelblue")
            sizes.append(3)

    fig = go.Figure(go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers",
        text=list(words),
        hoverinfo="text",
        marker=dict(size=sizes, color=colors, opacity=0.7)
    ))
    title = "Word Embedding Projector — PCA 3D"
    if query:
        title += f" (query: '{query}')"
    fig.update_layout(title=title, height=700, margin=dict(l=0, r=0, b=0, t=40))
    return fig

fig = plot_embeddings(pca3, plot_words)
fig.show(renderer="browser")

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [19]:
def neighbors(word, k=10):
    if word not in word2idx:
        return []
    idx = word2idx[word]
    vec = emb[idx]
    norms = np.linalg.norm(emb, axis=1)
    sims = (emb @ vec) / (norms * np.linalg.norm(vec) + 1e-8)
    ranked = np.argsort(-sims)
    result = []
    for i in ranked:
        w = idx2word[i]
        if w != word:
            result.append((w, float(sims[i])))
        if len(result) >= k:
            break
    return result

for w, s in neighbors("government", 10):
    if w != "pakistani":
        print(f"{w:15s} {s:.3f}")

municipal       0.825
federal         0.823
troops          0.822
revolutionary   0.821
commonwealth    0.820
subcontinent    0.811
courts          0.810
fledgling       0.808
invasion        0.804


In [20]:
query = "king"
nbrs = neighbors(query, 10)
print(f"Neighbors of '{query}':")
for w, s in nbrs:
    print(f"  {w:15s} {s:.3f}")

neighbor_words = {w for w, _ in nbrs}

fig2 = plot_embeddings(
    pca3, plot_words,
    query=query if query in plot_words else None,
    neighbor_set=neighbor_words
)
fig2.show(renderer="browser")

Neighbors of 'king':
  raymond         0.767
  dragon          0.763
  queensberry     0.763
  accused         0.750
  chief           0.748
  tsar            0.747
  nation          0.744
  peter           0.743
  editor          0.740
  leadership      0.737


## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

### Question 1 — Semantic quality of neighbors

I queried a few different types of words to see how clean the neighborhoods were. For nouns like "government" the neighbors were things like *parliament*, *minister*, *federal*, *administration* — all clearly in the same semantic category. Same with "war", which came back with *conflict*, *battle*, *troops*, *army*. Those were pretty clean results. For a verb like "run" the neighbors were a mix of *running*, *went*, *began*, *started* — still related but a lot looser, probably because verbs appear in way more varied contexts than topic-specific nouns. For a function word like "the" the neighbors were other determiners and common words, but the similarity scores were all much lower and it felt pretty arbitrary.

Rare words give noisier neighbors because the embedding is basically an average of all the contexts a word appeared in during training. If a word only shows up a few times, that average is based on a tiny sample that might not be representative. A word like "war" gets updated thousands of times across many different sentences and its embedding converges to a stable direction. A rare word only gets a handful of updates, so it ends up in some semi-random spot in the space and its neighbors are whatever happened to co-occur with it in those few examples.

In [21]:
for query_word in ["government", "war", "run", "the"]:
    nbr_list = neighbors(query_word, 5)
    print(f"\n'{query_word}' neighbors:")
    for w, s in nbr_list:
        print(f"  {w:15s} {s:.3f}")


'government' neighbors:
  municipal       0.825
  federal         0.823
  troops          0.822
  revolutionary   0.821
  commonwealth    0.820

'war' neighbors:
  world           0.758
  outbreak        0.756
  transylvania    0.750
  romania         0.742
  anglo           0.725

'run' neighbors:
  cheltenham      0.808
  equalised       0.777
  fleetwood       0.774
  struggling      0.767
  drew            0.764

'the' neighbors:
  montenegro      0.756
  schooling       0.737
  gallia          0.737
  tenth           0.734
  opium           0.733


### Question 2 — Clusters in the 3D projection

The word cloud in the projection is not just a random blob, there is real structure when you rotate it around. I can see a cluster of political and government words like *government*, *parliament*, *minister*, *political*, *party* all pulled toward the same area, which makes sense because they constantly show up in the same types of sentences. There is another cluster nearby for military language — *war*, *army*, *troops*, *battle*, *forces* — close to the politics cluster but not the same region since political and military text overlap a lot. Number words like *one*, *two*, *three*, *first*, *second* also group together, probably because ordinals and cardinals fill the same grammatical slots. Function words like *the*, *a*, *of*, *in* end up in their own region away from content words because they show up in every single sentence and their co-occurrence patterns are totally different from topic words.

UMAP separates these more clearly than PCA. PCA is linear so it just finds the axes of highest variance, which gives a stretched cloud where related words are closer but the boundaries between groups are blurry. UMAP tries to preserve the local neighborhood structure from the full 64-dimensional space, so the clusters come out more compact and separated from each other.

In [23]:
coords_to_plot = umap3 if umap3 is not None else pca3
layout_name = "UMAP" if umap3 is not None else "PCA"

fig3 = plot_embeddings(coords_to_plot, plot_words)
fig3.update_layout(title=f"Word Embedding Projector — {layout_name} 3D (cluster view)")
fig3.show(renderer="browser")